# 05 – Clustering
**Practical 16:** K-Means and hierarchical clustering to find patterns in meals. Clustering is unsupervised, so it does not use the `high_waste` label for grouping.

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # lets us import from the src/ folder
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
from scipy.cluster.hierarchy import linkage, dendrogram
from src.preprocessing import load_clean_data
from src.clustering import (CLUSTER_COLS, scale_for_clustering, elbow_and_silhouette,
                            fit_kmeans, pca_2d, cluster_profiles)

df = load_clean_data()
Xc = scale_for_clustering(df)
print("Columns used:", CLUSTER_COLS)

## Choosing k – elbow and silhouette

In [ ]:
ks, inertia, sil = elbow_and_silhouette(Xc)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(ks, inertia, marker="o"); axes[0].set_title("Elbow method"); axes[0].set_xlabel("k")
axes[1].plot(ks, sil, marker="o");     axes[1].set_title("Silhouette score"); axes[1].set_xlabel("k")
plt.tight_layout()
plt.savefig(FIG_DIR / "clustering_k_selection.png", dpi=120)
plt.show()

## K-Means

In [ ]:
K = 3   # chosen from the plots and the dendrogram below; change if your plots suggest otherwise
labels = fit_kmeans(Xc, K)

pts = pca_2d(Xc)
plt.figure(figsize=(6, 5))
plt.scatter(pts[:, 0], pts[:, 1], c=labels, cmap="viridis", s=30)
plt.title(f"K-Means clusters (k={K})"); plt.xlabel("PC1"); plt.ylabel("PC2")
plt.savefig(FIG_DIR / "clustering_kmeans.png", dpi=120)
plt.show()

### What does each cluster look like?

In [ ]:
profiles = cluster_profiles(df, labels)
print(profiles)
print("\nMeals per cluster:\n", pd.crosstab(labels, df["meal"].values))

Give each cluster a plain-language name (for example *busy and well-matched*, *over-cooked*, *quiet and tight*). Note that `plates_eaten` and `leftover_kg` are used for clustering, so these groups **describe** meals after the fact and do not predict anything.

## Hierarchical clustering (dendrogram)

In [ ]:
plt.figure(figsize=(10, 4))
dendrogram(linkage(Xc, method="ward"), truncate_mode="lastp", p=20)
plt.title("Hierarchical clustering (Ward linkage)")
plt.xlabel("Cluster (size)"); plt.ylabel("Distance")
plt.savefig(FIG_DIR / "clustering_dendrogram.png", dpi=120)
plt.show()